# MDAO Workflow: Constraint-Aware DOE, NSGA-II, and Sobol Sensitivity

This tutorial runs the full multidisciplinary design workflow on a comms array:

1. constraint-aware DOE to map the feasible trade space,
2. NSGA-II to find the true Pareto front (not a weighted-sum compromise),
3. Sobol indices to rank which variables actually drive the outputs,
4. an HTML report with an embedded interactive trade plot.

```bash
pip install "phased-array-systems[mdao,plotting]"
```

In [1]:
import warnings

warnings.filterwarnings("ignore")

from phased_array_systems.requirements import Requirement, RequirementSet
from phased_array_systems.scenarios import CommsLinkScenario
from phased_array_systems.trades import (
    BatchRunner,
    DesignSpace,
    extract_pareto,
    filter_feasible,
    generate_doe,
    optimize_pareto,
    sobol_sensitivity,
)

scenario = CommsLinkScenario(
    freq_hz=10e9,
    bandwidth_hz=10e6,
    range_m=100e3,
    required_snr_db=10.0,
)

requirements = RequirementSet(
    requirements=[
        Requirement(
            id="R1",
            name="Positive link margin",
            metric_key="link_margin_db",
            op=">=",
            value=0.0,
            severity="must",
        ),
    ],
)

## 1. Constraint-aware DOE

Array sizes obey a sub-array divisibility rule, so a plain box-sampled DOE wastes points on unbuildable architectures. `validate="architecture"` re-draws until every case constructs.

In [2]:
space = (
    DesignSpace(name="Comms MDAO")
    .add_variable("array.nx", "int", low=4, high=32)
    .add_variable("array.ny", "int", low=4, high=32)
    .add_variable("rf.tx_power_w_per_elem", "float", low=0.5, high=2.0)
)

doe = generate_doe(space, n_samples=200, seed=7, validate="architecture")
results = BatchRunner(scenario, requirements).run(doe)

n_errors = results["meta.error"].notna().sum() if "meta.error" in results else 0
print(f"{len(results)} cases, {n_errors} construction errors")
print("feasible:", len(filter_feasible(results, requirements)))

Running 200 cases (0 cached)


Completed 200 cases in 1.1s (0.006s/case)
200 cases, 0 construction errors
feasible: 191


## 2. NSGA-II Pareto front

`optimize_pareto` searches the same space directly. Must-severity requirements become inequality constraints handled by constraint domination — no penalty weight to tune — and mixed variable types (here integers and a float) are handled natively.

In [3]:
front = optimize_pareto(
    space,
    scenario,
    objectives=[("eirp_dbw", "maximize"), ("cost_usd", "minimize")],
    requirements=requirements,
    n_generations=30,
    pop_size=32,
    seed=7,
)
front[["array.nx", "array.ny", "rf.tx_power_w_per_elem", "eirp_dbw", "cost_usd"]].round(
    2
).sort_values("cost_usd")

,array.nx,array.ny,rf.tx_power_w_per_elem,eirp_dbw,cost_usd
1,8,8,2.0,43.10,6400.0
0,8,16,2.0,49.13,12800.0
2,32,8,2.0,55.15,25600.0
3,32,16,2.0,61.17,51200.0


Compare against the DOE-derived front at the same objectives:

In [4]:
doe_front = extract_pareto(
    filter_feasible(results, requirements),
    objectives=[("eirp_dbw", "maximize"), ("cost_usd", "minimize")],
)
print(f"DOE front: {len(doe_front)} points from {len(results)} evaluations")
print(f"NSGA-II front: {len(front)} points from ~{30 * 32} evaluations")

DOE front: 11 points from 200 evaluations
NSGA-II front: 4 points from ~960 evaluations


## 3. Sobol global sensitivity

Which variables actually drive the outputs? S1 is each variable's first-order share of output variance; ST includes its interactions. Constrained integers stay in `base_config` (variance-based methods need a rectangular sampled domain).

In [5]:
sens_space = (
    DesignSpace()
    .add_variable("rf.tx_power_w_per_elem", "float", low=0.5, high=4.0)
    .add_variable("rf.pa_efficiency", "float", low=0.2, high=0.5)
    .add_variable("rf.noise_figure_db", "float", low=1.0, high=6.0)
)

indices = sobol_sensitivity(
    sens_space,
    scenario,
    metric_keys=["link_margin_db", "prime_power_w"],
    base_config={"array.nx": 16, "array.ny": 16},
    n_base=128,
    seed=3,
)
indices.round(3)

Running 640 cases (0 cached)


Completed 640 cases in 3.2s (0.005s/case)


,parameter,metric,S1,S1_conf,ST,ST_conf
0,rf.tx_power_w_per_elem,link_margin_db,0.732,0.192,0.732,0.143
1,rf.pa_efficiency,link_margin_db,0.000,0.000,0.000,0.000
2,rf.noise_figure_db,link_margin_db,0.275,0.127,0.272,0.064
3,rf.tx_power_w_per_elem,prime_power_w,0.744,0.212,0.779,0.189
4,rf.pa_efficiency,prime_power_w,0.240,0.128,0.300,0.093
5,rf.noise_figure_db,prime_power_w,0.000,0.000,0.000,0.000


TX power dominates the link margin, with the noise figure second; PA efficiency has no effect on margin at all (it only spends power), which is exactly what the prime-power indices show from the other side.

## 4. Report with an embedded interactive plot

`HTMLReport` embeds a self-contained interactive Pareto section (plotly.js inlined — the file works offline) when two objectives are configured.

In [6]:
from phased_array_systems.reports import HTMLReport
from phased_array_systems.reports.generator import ReportConfig

config = ReportConfig(
    title="Comms MDAO study",
    objectives=[("cost_usd", "minimize"), ("eirp_dbw", "maximize")],
)
html = HTMLReport(config).generate(results)
print(f"report: {len(html) / 1e6:.1f} MB, interactive section:", "Interactive Trade-off" in html)

report: 4.9 MB, interactive section: True


## Where to go next

- `pasys optimize --method nsga2` and `pasys sensitivity --sens-method sobol` run steps 2-3 from the command line.
- The DBF architecture tutorial applies this workflow to the digitization-level trade.
- The user guide's MDAO Tools page documents every function used here.